In [ ]:
"""
Stage 1 - CIC-IDS 2017 Data Loader and Text Representation Builder
"""

import json
import re
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

#  Paths 
DATA_DIR   = Path("../data/cicids")
STIX_FILE  = Path("../data/attck/enterprise-attack.json")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load MITRE ATT&CK STIX and build lookup 
def load_attck_stix(stix_path):
    """
    Parse the official MITRE ATT&CK STIX bundle and return a dict:
        technique_id -> {name, tactic, description, url}
    Source: https://github.com/mitre/cti/blob/master/enterprise-attack/enterprise-attack.json
    """
    print("Loading MITRE ATT&CK STIX bundle...")
    with open(stix_path, "r", encoding="utf-8") as f:
        bundle = json.load(f)

    techniques = {}
    for obj in bundle.get("objects", []):
        if obj.get("type") != "attack-pattern":
            continue
        if obj.get("revoked", False) or obj.get("deprecated", False):
            continue

        # Extract technique ID 
        tech_id = None
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack":
                tech_id = ref.get("external_id")
                break
        if not tech_id:
            continue

        # Extract tactics
        tactics = []
        for phase in obj.get("kill_chain_phases", []):
            if phase.get("kill_chain_name") == "mitre-attack":
                # Convert kebab-case to Title Case
                tactic = phase["phase_name"].replace("-", " ").title()
                tactics.append(tactic)

        techniques[tech_id] = {
            "name":        obj.get("name", ""),
            "tactic":      tactics[0] if tactics else "Unknown",
            "all_tactics": tactics,
            "description": obj.get("description", "")[:300],
        }

    print(f"  Loaded {len(techniques)} active ATT&CK techniques from STIX bundle.")
    return techniques

# Define mappings using official IDs 
def build_label_map(techniques):
    """
    Map CIC-IDS 2017 traffic labels to official ATT&CK technique IDs.
    """
    # Raw label -> technique ID
    # Labels use regex to handle encoding variants of the dash character
    raw_map = {
        "FTP-Patator":     "T1110.001",
        "SSH-Patator":     "T1110.001",
        "DoS slowloris":   "T1499.001",
        "DoS Slowhttptest":"T1499.001",
        "DoS Hulk":        "T1499.001",
        "DoS GoldenEye":   "T1499.001",
        "Heartbleed":      "T1190",
        "Web Attack Brute Force": "T1110.001",
        "Web Attack XSS":         "T1059.007",
        "Web Attack Sql Injection":"T1190",
        "Infiltration":    "T1105",
        "Bot":             "T1071.001",
        "PortScan":        "T1046",
        "DDoS":            "T1498.001",
        "BENIGN":          None,
    }

    label_map = {}
    for label, tech_id in raw_map.items():
        if tech_id is None:
            label_map[label] = {
                "technique_id":   "BENIGN",
                "technique_name": "Benign Traffic",
                "tactic":         "None",
            }
        elif tech_id in techniques:
            label_map[label] = {
                "technique_id":   tech_id,
                "technique_name": techniques[tech_id]["name"],
                "tactic":         techniques[tech_id]["tactic"],
            }
        else:
            print(f"  WARNING: {tech_id} not found in STIX bundle — check ID")
            label_map[label] = {
                "technique_id":   tech_id,
                "technique_name": "Unknown",
                "tactic":         "Unknown",
            }

    return label_map

def normalise_label(raw_label):
    """
    Fix encoding issues and normalise label strings.
    'Web Attack â€" Brute Force' / 'Web Attack ? Brute Force'
    all become 'Web Attack Brute Force'.
    """
    if not isinstance(raw_label, str):
        return "BENIGN"
    # Remove any non-ASCII dash variants and the word 'Attack -'
    label = re.sub(r'[^\x00-\x7F]+', '', raw_label)   # strip non-ASCII
    label = re.sub(r'\s*[-–—]\s*', ' ', label)         # replace dashes
    label = re.sub(r'\s+', ' ', label).strip()          # collapse spaces
    return label

# Build rich text representation 
def build_text(row):
    """
    Build a semantically rich plain-English sentence from flow features.
    Uses only columns confirmed present in this CIC-IDS version.
    Designed for all-MiniLM-L6-v2 sentence embedding.
    """
    try:
        label      = str(row.get("Label", "BENIGN")).strip()
        tech_id    = str(row.get("attck_technique_id", "")).strip()
        tactic     = str(row.get("attck_tactic", "")).strip()
        dst_port   = row.get("Destination Port", 0)
        duration   = row.get("Flow Duration", 0)
        fwd_pkts   = row.get("Total Fwd Packets", 0)
        bwd_pkts   = row.get("Total Backward Packets", 0)
        pkt_mean   = row.get("Packet Length Mean", 0)
        pkt_std    = row.get("Packet Length Std", 0)
        flow_bps   = row.get("Flow Bytes/s", 0)
        flow_pps   = row.get("Flow Packets/s", 0)
        syn_flag   = int(row.get("SYN Flag Count", 0))
        fin_flag   = int(row.get("FIN Flag Count", 0))
        rst_flag   = int(row.get("RST Flag Count", 0))
        psh_flag   = int(row.get("PSH Flag Count", 0))
        ack_flag   = int(row.get("ACK Flag Count", 0))
        init_win_f = row.get("Init_Win_bytes_forward", 0)
        init_win_b = row.get("Init_Win_bytes_backward", 0)
        idle_mean  = row.get("Idle Mean", 0)
        active_mean= row.get("Active Mean", 0)

        # Describe port context
        port_context = {
            21: "FTP control",
            22: "SSH",
            23: "Telnet",
            25: "SMTP",
            53: "DNS",
            80: "HTTP",
            443: "HTTPS",
            3306: "MySQL",
            3389: "RDP",
            8080: "HTTP-alt",
        }.get(int(dst_port) if str(dst_port).isdigit() else 0,
              f"port {dst_port}")

        # Describe TCP flag pattern
        flags = []
        if syn_flag > 0: flags.append("SYN")
        if fin_flag > 0: flags.append("FIN")
        if rst_flag > 0: flags.append("RST")
        if psh_flag > 0: flags.append("PSH")
        if ack_flag > 0: flags.append("ACK")
        flag_str = "+".join(flags) if flags else "no flags"

        # Describe flow rate
        try:
            bps_val = float(flow_bps)
            if bps_val > 1_000_000:
                rate_str = f"{bps_val/1_000_000:.1f} Mbps"
            elif bps_val > 1_000:
                rate_str = f"{bps_val/1_000:.1f} Kbps"
            else:
                rate_str = f"{bps_val:.1f} bps"
        except Exception:
            rate_str = "unknown rate"

        text = (
            f"Network flow targeting {port_context} destination. "
            f"Duration {int(float(duration))} microseconds. "
            f"Forward packets {int(float(fwd_pkts))}, "
            f"backward packets {int(float(bwd_pkts))}. "
            f"Mean packet length {round(float(pkt_mean), 1)} bytes "
            f"(std {round(float(pkt_std), 1)}). "
            f"Flow rate {rate_str}. "
            f"TCP flags observed: {flag_str}. "
            f"Initial window bytes forward {int(float(init_win_f))}, "
            f"backward {int(float(init_win_b))}. "
            f"Mean active time {round(float(active_mean))} microseconds, "
            f"mean idle time {round(float(idle_mean))} microseconds. "
            f"Traffic classified as {label}."
            + (f" ATT&CK technique {tech_id} ({tactic})." 
               if tech_id and tech_id != "BENIGN" else "")
        )
        return text

    except Exception as e:
        return f"Network flow classified as {row.get('Label', 'BENIGN')}."

# Main pipeline
def load_and_process():
    # Load STIX and build verified label map
    techniques = load_attck_stix(STIX_FILE)
    label_map  = build_label_map(techniques)

    print("\nVerified ATT&CK label mappings:")
    for lbl, info in label_map.items():
        if info["technique_id"] != "BENIGN":
            print(f"  {lbl:35s} -> {info['technique_id']:12s} "
                  f"{info['technique_name']} [{info['tactic']}]")

    all_frames = []
    available  = sorted(DATA_DIR.glob("*.csv"))
    print(f"\nFound {len(available)} CSV files in {DATA_DIR}")

    for filepath in available:
        fname = filepath.name.lower()

        # Determine day
        if   "tuesday"   in fname: day = "Tuesday"
        elif "thursday"  in fname: day = "Thursday"
        elif "friday"    in fname: day = "Friday"
        else:
            print(f"\nSkipping {filepath.name}")
            continue

        print(f"\nLoading {filepath.name} ({day})...")

        try:
            df = pd.read_csv(
                filepath,
                low_memory=False,
                encoding="latin-1",      # handles the dash encoding issue
            )
            df.columns = df.columns.str.strip()
            print(f"  Raw rows: {len(df):,}")

            # Normalise Label column
            df["Label"] = df["Label"].apply(normalise_label)

            # Filter: all attacks + 10% benign
            attacks = df[df["Label"] != "BENIGN"].copy()
            benign  = df[df["Label"] == "BENIGN"].sample(
                frac=0.10, random_state=42
            ).copy()
            df = pd.concat([attacks, benign], ignore_index=True)

            print(f"  Label counts after normalise:")
            print(df["Label"].value_counts().to_string())

            # Add ATT&CK columns from verified STIX map
            def get_field(lbl, field):
                # Try exact match first, then normalised
                info = label_map.get(lbl)
                if info is None:
                    norm = normalise_label(lbl)
                    info = label_map.get(norm,{
                        "technique_id":"UNMAPPED",
                        "technique_name": lbl,
                        "tactic":"Unknown"
                    })
                return info[field]

            df["attck_technique_id"]   = df["Label"].apply(
                lambda x: get_field(x, "technique_id"))
            df["attck_technique_name"] = df["Label"].apply(
                lambda x: get_field(x, "technique_name"))
            df["attck_tactic"]         = df["Label"].apply(
                lambda x: get_field(x, "tactic"))
            df["day"]                  = day

            # Clean numeric columns
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            df[numeric_cols] = df[numeric_cols].replace(
                [np.inf, -np.inf], np.nan
            ).fillna(0)

            # Build text representations
            tqdm.pandas(desc=f"  Building text [{day}]")
            df["alert_text"] = df.progress_apply(build_text, axis=1)

            all_frames.append(df)

        except Exception as e:
            print(f"  ERROR: {e}")
            import traceback; traceback.print_exc()
            continue

    if not all_frames:
        print("ERROR: No files loaded.")
        return

    # Combine all days
    combined = pd.concat(all_frames, ignore_index=True)

    # Keep useful columns
    keep = [
        "Destination Port", "Flow Duration",
        "Total Fwd Packets", "Total Backward Packets",
        "Packet Length Mean", "Packet Length Std",
        "Flow Bytes/s", "Flow Packets/s",
        "SYN Flag Count", "FIN Flag Count", "RST Flag Count",
        "PSH Flag Count", "ACK Flag Count",
        "Init_Win_bytes_forward", "Init_Win_bytes_backward",
        "Active Mean", "Idle Mean",
        "Label", "attck_technique_id", "attck_technique_name",
        "attck_tactic", "day", "alert_text"
    ]
    keep     = [c for c in keep if c in combined.columns]
    combined = combined[keep].copy()
    combined.reset_index(drop=True, inplace=True)
    combined.index.name = "alert_id"

    # Save
    output_path = OUTPUT_DIR / "cicids_processed.csv"
    combined.to_csv(output_path)

    # Summary statistics
    print(f"\n{'='*85}")
    print(f"Total alerts          : {len(combined):,}")
    print(f"Attack alerts         : {len(combined[combined['Label']!='BENIGN']):,}")
    print(f"Benign alerts (10%)   : {len(combined[combined['Label']=='BENIGN']):,}")
    print(f"ATT&CK tactics covered: {combined['attck_tactic'].nunique()}")
    print(f"Unique attack types   : {combined[combined['Label']!='BENIGN']['Label'].nunique()}")
    print(f"\nTactic distribution:")
    print(combined['attck_tactic'].value_counts().to_string())
    print(f"\nATT&CK technique distribution:")
    print(combined['attck_technique_id'].value_counts().to_string())
    unmapped = combined[combined['attck_technique_id']=='UNMAPPED']
    if len(unmapped) > 0:
        print(f"\nWARNING: {len(unmapped)} unmapped labels:")
        print(unmapped['Label'].value_counts().to_string())
    print(f"\nSample alert texts:")
    for i, row in combined[combined['Label']!='BENIGN'].head(3).iterrows():
        print(f"\n  [{row['Label']} | {row['attck_technique_id']} | {row['attck_tactic']}]")
        print(f"  {row['alert_text']}")
    print(f"\nSaved to: {output_path}")

if __name__ == "__main__":
    load_and_process()

Loading MITRE ATT&CK STIX bundle...
  Loaded 703 active ATT&CK techniques from STIX bundle.

Verified ATT&CK label mappings:
  FTP-Patator                         -> T1110.001    Password Guessing [Credential Access]
  SSH-Patator                         -> T1110.001    Password Guessing [Credential Access]
  DoS slowloris                       -> T1499.001    OS Exhaustion Flood [Impact]
  DoS Slowhttptest                    -> T1499.001    OS Exhaustion Flood [Impact]
  DoS Hulk                            -> T1499.001    OS Exhaustion Flood [Impact]
  DoS GoldenEye                       -> T1499.001    OS Exhaustion Flood [Impact]
  Heartbleed                          -> T1190        Exploit Public-Facing Application [Initial Access]
  Web Attack Brute Force              -> T1110.001    Password Guessing [Credential Access]
  Web Attack XSS                      -> T1059.007    JavaScript [Execution]
  Web Attack Sql Injection            -> T1190        Exploit Public-Facing Applicati

  Building text [Friday]: 100%|██████████| 137799/137799 [00:03<00:00, 37705.04it/s]



Loading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv (Friday)...
  Raw rows: 286,467
  Label counts after normalise:
Label
PortScan    158930
BENIGN       12754


  Building text [Friday]: 100%|██████████| 171684/171684 [00:06<00:00, 26895.06it/s]



Loading Friday-WorkingHours-Morning.pcap_ISCX.csv (Friday)...
  Raw rows: 191,033
  Label counts after normalise:
Label
BENIGN    18907
Bot        1966


...


Loading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv (Thursday)...
  Raw rows: 170,366
  Label counts after normalise:
Label
BENIGN                      16819
Web Attack Brute Force       1507
Web Attack XSS                652
Web Attack Sql Injection       21


  Building text [Thursday]: 100%|██████████| 18999/18999 [00:00<00:00, 40072.75it/s]



Loading Tuesday-WorkingHours.pcap_ISCX.csv (Tuesday)...
  Raw rows: 445,909
  Label counts after normalise:
Label
BENIGN         43207
FTP Patator     7938
SSH Patator     5897


  Building text [Tuesday]: 100%|██████████| 57042/57042 [00:01<00:00, 36761.04it/s]



Skipping Wednesday-workingHours.pcap_ISCX.csv

Total alerts          : 435,290
Attack alerts         : 304,974
Benign alerts (10%)   : 130,316
ATT&CK tactics covered: 8
Unique attack types   : 9

Tactic distribution:
attck_tactic
Discovery              158930
None                   130316
Impact                 128027
Unknown                 13835
Command And Control      2002
Credential Access        1507
Execution                 652
Initial Access             21

ATT&CK technique distribution:
attck_technique_id
T1046        158930
BENIGN       130316
T1498.001    128027
UNMAPPED      13835
T1071.001      1966
T1110.001      1507
T1059.007       652
T1105            36
T1190            21

Label
FTP Patator    7938
SSH Patator    5897

Sample alert texts:

  [DDoS | T1498.001 | Impact]
  Network flow targeting HTTP destination. Duration 1293792 microseconds. Forward packets 3, backward packets 7. Mean packet length 1057.5 bytes (std 1853.4). Flow rate 9.0 Kbps. TCP flags observed: 